# Phase A: Data Preparation for CLIP ViT-B/32 on CelebA
## Steps 1-3: Load dataset & attributes (from data_loader), load evaluation JSON, sanity checks

## Step 1: Load CelebA Dataset (test split only)

**Why:** We need the actual images in a standardized format with CLIP-compatible preprocessing.
- Resize to 224×224 (CLIP ViT-B/32's expected input)
- Normalize using CLIP's standard statistics
- Load only test split (~19,962 images) to evaluate on unseen data


In [ ]:
import sys
import json
from pathlib import Path

import torch

# --- Make src/ importable regardless of where Jupyter was launched -----------
# Find the project root by walking up until we see the src/ + Evaluation/ dirs.
_here = Path.cwd()
project_root = next(
    (p for p in [_here, *_here.parents] if (p / 'src').is_dir() and (p / 'Evaluation').is_dir()),
    _here,
)
src_dir = project_root / 'src'
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

from data_loader import load_celeba_dataset, load_attributes, ATTRIBUTE_NAMES, setup_frozen_db

print(f"Project root: {project_root}")
print()

# Ensure frozen DB exists (built into artifacts/ on first run)
setup_frozen_db()
print()

# Load CelebA dataset and attributes
celeba = load_celeba_dataset()
celeba_attrs = load_attributes()

print(f"[OK] CelebA test set loaded successfully")
print(f"  Dataset size: {len(celeba)} images")
print(f"  Attributes shape: {celeba_attrs.shape}")
print(f"  Attributes dtype: {celeba_attrs.dtype}")
print()

## Step 2: Load Evaluation JSON

**Why:** This JSON defines the ground truth benchmark for evaluating retrieval quality.

Each query specifies:
- **Sources:** All images matching the query condition (e.g., 4,786 smiling images for "+Smiling")
- **Valid targets:** For each source, the list of other images that are similar by attribute (Hamming distance ≤ 2) AND match the query condition

**How it's built:**
1. For each of the 4,786 smiling images (source):
   - Find all similar images using Hamming distance ≤ 2
   - Filter to only those that ALSO have "+Smiling" attribute
   - Record this list as valid targets for this source
2. Average: (targets for source1 + targets for source2 + ... ) / 4,786 = **26.0 avg targets**

**Later evaluation (Phase B):**
- Extract CLIP embeddings for all test images
- For each source, retrieve top-26 similar images using CLIP
- Check: How many of CLIP's retrievals match the ground truth targets?
- Score: Recall/Precision = (correct retrievals / expected targets)

In [ ]:
eval_json_path = project_root / 'Evaluation' / 'celeba_evaluation.json'
print(f"Reading from: {eval_json_path}")

with open(eval_json_path, 'r') as f:
    queries = json.load(f)

print(f"  Total queries: {len(queries)}")

# Display summary of each query
for i, query_dict in enumerate(queries):
    query_str = query_dict['query']
    ground_truth = query_dict['ground_truth']
    num_sources = len(ground_truth)

    # Count total targets
    total_targets = sum(len(targets) for targets in ground_truth.values())
    avg_targets = total_targets / num_sources if num_sources > 0 else 0

    print(f"  Query {i}: '{query_str}'")
    print(f"    Sources: {num_sources}, Avg targets per source: {avg_targets:.1f}")

## Step 3: Sanity Check (CRITICAL)

**Why:** Index consistency is fragile — dataset indices don't match filenames.
**One failure here means everything downstream is silently wrong.**

These checks live in [`src/sanity.py`](../src/sanity.py) as *asserting* functions (not prints),
so a broken assumption fails loudly here — before any retrieval method produces a wrong score.
`run_all_checks()` verifies, in order:

1. **Indexing** (`CONTRACT.md` §0) — `celeba.filename[13] == "182651.jpg"` and `N == 19962`.
2. **Attribute tensor** (`CONTRACT.md` §2) — `[N, 40]` float32, values in `{0.0, 1.0}`, row-aligned to the dataset index.
3. **Ground-truth viability** (`ROADMAP.md`) — every source has ≥ 5 valid targets, all indices in range, no source in its own target list.

In [ ]:
# All Day-1 CRITICAL checks, as hard assertions. Raises on the first failure.
from sanity import run_all_checks

summary = run_all_checks()   # prints a green-light report; raises AssertionError if anything is off
summary